# 03 — Data cleaning: FAERS GLP-1 reports
**Owner:** Brian  ·  **Output:** Logistic Regression Model Based on EDA

| Step | Details            |
|------|--------------------|
| 0    | Setup              |
| 1    | Load the Data      |
| 2    | Data Split (80:20) |
| 3    | Preprocessing      |
| 4    | Model              |
| 5    | Evaluation         |

## Step 0 -- Setup

In [ ]:
# Libraries
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

## Step 1 -- Load the Data

In [ ]:
# Load the data
df = pd.read_parquet('../data/03_model/glp1_model_ready.parquet')
df.head()

In [ ]:
# Drop ID columns
dropped_columns = [
    'primaryid',
    'caseid'
]
df = df.drop(columns=dropped_columns)
df.columns

In [ ]:
# Data type conversion (Based on decision table)
df.info()

In [ ]:
# Define categorical and numeric columns
cat_cols = [
    'drug',
    'age_group',
    'sex',
    'reporter_type',
    'indication_group',
    'route_group'
]
num_cols = df.columns.difference(cat_cols).tolist() # All other categories are numeric
num_cols.remove('is_serious')
df[cat_cols] = df[cat_cols].astype('category')
df.info()

## Step 2 -- Data Split (80:20)

In [ ]:
# Split the data into training and test sets
y = df['is_serious']
X = df.drop(columns=['is_serious'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## Step 3 -- Preprocessing

In [ ]:
# Preprocessing pipelines for numeric and categorical data
num_pl = Pipeline([
    ('scaler', RobustScaler())
])
cat_pl = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', num_pl, num_cols),
    ('cat', cat_pl, cat_cols)
])

## Step 4 -- Model

In [ ]:
# Build the full pipeline with preprocessing and classifier
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
full_pipeline.fit(X_train, y_train)
predictions = full_pipeline.predict(X_test)

## Step 5 -- Evaluate

In [ ]:
# Scores
print(f'Classification Report:\n{classification_report(y_test, predictions)}')
print(f'Confusion Matrix:\n{confusion_matrix(y_test, predictions)}')
print(f'Average Precision Score: {average_precision_score(y_test, predictions)}')
print(f'ROC AUC Score: {roc_auc_score(y_test, predictions)}')